# Export Window Mean-Beat Plots

Load a session mean-beats pickle and export one overlay PNG per window.


In [6]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np

PICKLE_NAME = "session_mean_beats_508_09Apr26_0252H.pkl"

candidate_base_dirs = [Path.cwd(), Path.cwd() / "signal-processing-intense"]
BASE_DIR = next((candidate for candidate in candidate_base_dirs if (candidate / PICKLE_NAME).exists()), candidate_base_dirs[-1])
PICKLE_FILE = BASE_DIR / PICKLE_NAME

# Export to <base_dir>/plots/<session_file_stem>/window1.png, ...
OUTPUT_ROOT = BASE_DIR / "plots"

# Fixed plot size for every PNG. Keep this modest so files stay small.
FIGSIZE = (5.6, 4.2)
DPI = 100
LINEWIDTH = 1.4
TITLE_FONTSIZE = 12
LABEL_FONTSIZE = 10
COLORS = {"CH2": "#1f77b4", "CH3": "#ff7f0e", "CH4": "#2ca02c"}

session_bin_candidates = sorted(BASE_DIR.glob("session_*.bin"))
if session_bin_candidates:
    SESSION_FILE = session_bin_candidates[0]
else:
    SESSION_FILE = Path("session")

OUTPUT_DIR = OUTPUT_ROOT / SESSION_FILE.stem
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base dir: {BASE_DIR.resolve()}")
print(f"Pickle file: {PICKLE_FILE}")
print(f"Session file: {SESSION_FILE.name}")
print(f"Output dir: {OUTPUT_DIR}")


Base dir: C:\src\capstone-ecgapp\signal-processing-intense
Pickle file: c:\src\capstone-ecgapp\signal-processing-intense\session_mean_beats_508_09Apr26_0252H.pkl
Session file: session_05Apr26_0459H.bin
Output dir: c:\src\capstone-ecgapp\signal-processing-intense\plots\session_05Apr26_0459H


In [ ]:
with PICKLE_FILE.open("rb") as f:
    session_mean_beats_payload = pickle.load(f)

windows = session_mean_beats_payload.get("windows", {})
window_keys = sorted(windows.keys(), key=lambda value: int(value))

def _get_epoch_axis(window_payload: dict) -> np.ndarray:
    epoch_axis = np.asarray(window_payload.get("epoch_axis", []), dtype=float)
    if epoch_axis.size > 0:
        return epoch_axis
    ch2 = np.asarray(window_payload["mean_beats"]["CH2"], dtype=float)
    return np.arange(len(ch2), dtype=float)

saved_files = []

for ordinal_index, window_key in enumerate(window_keys, start=1):
    window_payload = windows[window_key]
    epoch_axis = _get_epoch_axis(window_payload)
    ch2 = np.asarray(window_payload["mean_beats"]["CH2"], dtype=float)
    ch3 = np.asarray(window_payload["mean_beats"]["CH3"], dtype=float)
    ch4 = np.asarray(window_payload["mean_beats"]["CH4"], dtype=float)

    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    ax.plot(epoch_axis[:len(ch2)], ch2, color=COLORS["CH2"], linewidth=LINEWIDTH, label="CH2")
    ax.plot(epoch_axis[:len(ch3)], ch3, color=COLORS["CH3"], linewidth=LINEWIDTH, label="CH3")
    ax.plot(epoch_axis[:len(ch4)], ch4, color=COLORS["CH4"], linewidth=LINEWIDTH, label="CH4")

    stacked = np.concatenate([ch2, ch3, ch4]) if len(ch2) and len(ch3) and len(ch4) else np.array([0.0])
    y_min = float(np.nanmin(stacked))
    y_max = float(np.nanmax(stacked))
    y_span = y_max - y_min
    y_pad = 0.1 * y_span if y_span > 0 else 1.0
    ax.set_ylim(y_min - y_pad, y_max + y_pad)

    ax.set_title(f"Window {ordinal_index}", fontsize=TITLE_FONTSIZE)
    ax.set_xlabel("Epoch axis", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel("Amplitude (mV)", fontsize=LABEL_FONTSIZE)
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right", fontsize=9)
    fig.tight_layout()

    output_path = OUTPUT_DIR / f"window{ordinal_index}.png"
    fig.savefig(output_path, facecolor="white")
    plt.close(fig)
    saved_files.append(output_path)

print(f"Saved {len(saved_files)} PNGs to {OUTPUT_DIR}")
for output_path in saved_files[:5]:
    size_kb = output_path.stat().st_size / 1024
    print(f"{output_path.name} | {size_kb:.1f} KB")


Saved 508 PNGs to c:\src\capstone-ecgapp\signal-processing-intense\plots\session_05Apr26_0459H
window1.png | 34.5 KB
window2.png | 33.6 KB
window3.png | 33.9 KB
window4.png | 33.8 KB
window5.png | 34.1 KB


: 